# 1_1 Segmentação e Limpeza

Realiza **segmentação e limpeza** do corpus de notícias brasileiras.

> **Corpus:** 32 notícias sobre economia, política e sociedade.
>
> **Funciona com ou sem Google Drive** — se o Drive não estiver disponível, usa o corpus embutido automaticamente.

## 1 Preparação do Ambiente

In [ ]:
import time
inicio_processamento = time.time()
print('Início do processamento registrado.')

Início do processamento registrado.


## 2 Carregamento do Corpus

In [ ]:
import sys, os, re, time, logging
import pandas as pd
from collections import Counter

logging.basicConfig(format="%(asctime)s : %(levelname)s : %(message)s")
logger = logging.getLogger(); logger.setLevel(logging.INFO)

# Detecta se está no Google Colab
IN_COLAB = "google.colab" in sys.modules

DIRETORIO_NOTEBOOK = "SRI"
NOME_ARQUIVO = "documentos.csv"

if IN_COLAB:
    # Tenta carregar do Google Drive
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DIRETORIO_DRIVE = "/content/drive/MyDrive/Colab Notebooks/" + DIRETORIO_NOTEBOOK + "/data/"
        DIRETORIO_LOCAL = "/content/" + DIRETORIO_NOTEBOOK + "/"
        os.makedirs(DIRETORIO_LOCAL, exist_ok=True)
        import shutil
        shutil.copy(DIRETORIO_DRIVE + NOME_ARQUIVO, DIRETORIO_LOCAL + NOME_ARQUIVO)
        logging.info("Arquivo carregado do Google Drive.")
    except Exception:
        # Se não tiver Drive, usa corpus embutido
        logging.info("Drive não disponível. Usando corpus embutido.")
        DIRETORIO_LOCAL = "/content/" + DIRETORIO_NOTEBOOK + "/"
        os.makedirs(DIRETORIO_LOCAL, exist_ok=True)
else:
    DIRETORIO_LOCAL = "./data/"
    os.makedirs(DIRETORIO_LOCAL, exist_ok=True)

# Corpus embutido (fallback)
_documentos = [
    "O presidente Luiz Inácio Lula da Silva assinou hoje em Brasília um pacote de medidas para estimular a economia brasileira, incluindo investimentos em infraestrutura e geração de empregos.",
    "O Banco Central do Brasil decidiu manter a taxa Selic em 10,5% ao ano, segundo comunicado divulgado pelo Comitê de Política Monetária nesta quarta-feira.",
    "A Petrobras anunciou lucro líquido de R$ 23 bilhões no primeiro trimestre deste ano, superando as expectativas dos analistas de mercado.",
    "O ministro da Fazenda, Fernando Haddad, apresentou ao Congresso Nacional o novo arcabouço fiscal, que prevê meta de déficit zero para 2025.",
    "A Bolsa de Valores de São Paulo, a B3, registrou alta de 1,2% nesta sexta-feira, impulsionada pelo desempenho positivo das ações do setor bancário.",
    "O governador de São Paulo, Tarcísio de Freitas, anunciou a privatização de mais quatro empresas estatais do estado nos próximos dois anos.",
    "O Instituto Brasileiro de Geografia e Estatística divulgou que a taxa de desemprego caiu para 7,8% no segundo trimestre, menor nível desde 2014.",
    "A Vale informou que a produção de minério de ferro atingiu recorde histórico no terceiro trimestre, com 89,4 milhões de toneladas métricas.",
    "O Supremo Tribunal Federal concluiu o julgamento sobre a constitucionalidade da reforma tributária aprovada pelo Congresso Nacional.",
    "O presidente do Banco Central, Roberto Campos Neto, alertou sobre os riscos fiscais que podem comprometer a estabilidade da moeda nacional.",
    "A empresa de tecnologia Totvs anunciou a aquisição de uma startup de inteligência artificial por R$ 450 milhões no Rio de Janeiro.",
    "O Senado Federal aprovou por 52 votos a 19 a proposta de emenda constitucional que modifica as regras do sistema previdenciário.",
    "A Embraer fechou contrato de US$ 2 bilhões com companhia aérea americana para fornecimento de aeronaves E2 nos próximos cinco anos.",
    "O ministro da Educação anunciou a expansão do programa Bolsa Família para famílias com crianças em idade escolar em todo o território nacional.",
    "A Agência Nacional de Vigilância Sanitária aprovou novos medicamentos genéricos para o tratamento de doenças crônicas no Brasil.",
    "O prefeito de São Paulo, Ricardo Nunes, inaugurou o trecho norte da Linha 6 do metrô, que liga a estação Brasilândia ao centro da cidade.",
    "O Brasil registrou superávit comercial de US$ 8,4 bilhões em setembro, impulsionado pelas exportações de soja e petróleo.",
    "A Câmara dos Deputados aprovou em primeiro turno a reforma administrativa, que altera as regras do serviço público federal.",
    "O Conselho Nacional de Justiça publicou resolução que moderniza o sistema de processos digitais no Judiciário brasileiro.",
    "A Ambev divulgou crescimento de 15% nas vendas no mercado brasileiro, atribuído ao aumento do consumo nas regiões Norte e Nordeste.",
    "O governo federal lançou o programa Minha Casa Minha Vida ampliado, com meta de construir 2 milhões de habitações até 2026.",
    "A Agência Nacional de Telecomunicações anunciou o leilão de frequências 5G para as cidades de médio porte do interior do Brasil.",
    "O Tribunal de Contas da União apontou irregularidades em contratos de obras do Departamento Nacional de Infraestrutura de Transportes.",
    "A Sabesp, empresa de saneamento de São Paulo, informou que atingiu a meta de universalização do tratamento de esgoto no estado.",
    "O presidente da Câmara, Arthur Lira, pautou para votação o projeto de lei sobre a tributação de plataformas digitais internacionais.",
    "O Brasil sediará a Conferência das Nações Unidas sobre Mudanças Climáticas, a COP30, na cidade de Belém do Pará em novembro de 2025.",
    "A Fiocruz anunciou o desenvolvimento de nova vacina contra a dengue em parceria com universidades federais do Rio de Janeiro e São Paulo.",
    "O Ministério do Trabalho publicou portaria que estabelece novas regras para o teletrabalho e o trabalho híbrido no setor privado.",
    "A Receita Federal do Brasil arrecadou R$ 218 bilhões em agosto, recorde histórico para o mês, superando a previsão do governo federal.",
    "O Banco Nacional de Desenvolvimento Econômico e Social aprovou financiamento de R$ 12 bilhões para projetos de energia renovável no Nordeste.",
    "A ministra do Meio Ambiente, Marina Silva, assinou decreto que amplia em 2 milhões de hectares a área de proteção ambiental na Amazônia.",
    "O Instituto Nacional do Seguro Social informou que o pagamento do 13º salário dos aposentados será antecipado para o mês de agosto.",
]

# Salva o CSV localmente se não existir
csv_path = DIRETORIO_LOCAL + NOME_ARQUIVO
if not os.path.exists(csv_path):
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("id;texto\n")
        for i, doc in enumerate(_documentos, 1):
            f.write(f'{i};"{doc}"\n')

# Lê o CSV
NOME_ARQUIVO_DATASET = NOME_ARQUIVO
df_dataset = pd.read_csv(csv_path, sep=";", encoding="UTF-8")
documentos = df_dataset["texto"].tolist()
logging.info(f"TERMINADO ORIGINAIS: {len(df_dataset)}.")
print(f"\nCorpus carregado: {len(documentos)} documentos.")
df_dataset.head(3)


Corpus carregado: 32 documentos.


id,texto
1,O presidente Luiz Inácio Lula da Silva assinou hoje em Brasí...
2,"O Banco Central do Brasil decidiu manter a taxa Selic em 10,..."
3,A Petrobras anunciou lucro líquido de R$ 23 bilhões no prime...


## 3 Funções de Limpeza

In [ ]:
import re

STOPWORDS_PT = set([
    'a','ao','aos','as','até','com','como','da','das','de','do','dos',
    'e','ela','elas','ele','eles','em','entre','essa','esse','esta','este',
    'eu','foi','foram','há','isso','já','mais','mas','me','muito','na','nas',
    'não','nos','nós','o','os','ou','para','pela','pelas','pelo','pelos','por',
    'qual','quando','que','quem','se','sem','seu','seus','sua','suas','são',
    'também','um','uma','umas','uns','você','à','às','é','ser','ter','sobre',
    'após','desde','durante','segundo','menos','além','dentro','fora',
])

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = re.sub(r'\d+', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def tokenizar(texto):
    return texto.split()

def remover_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS_PT and len(t) > 2]

print(f'Stopwords: {len(STOPWORDS_PT)} | Funções definidas!')

Stopwords: 54 | Funções definidas!


## 4 Processamento do Corpus

In [ ]:
docs_processados = []
for i, doc in enumerate(documentos):
    limpo = limpar_texto(doc)
    tokens = tokenizar(limpo)
    sem_sw = remover_stopwords(tokens)
    docs_processados.append({
        'id': i + 1, 'original': doc,
        'limpo': limpo, 'tokens': tokens, 'tokens_sem_sw': sem_sw
    })

print(f"Documentos processados: {len(docs_processados)}")
print(f"\nExemplo - Documento 1:")
print(f"  Original : {docs_processados[0]['original'][:70]}...")
print(f"  Tokens   : {docs_processados[0]['tokens'][:7]}")
print(f"  Sem SW   : {docs_processados[0]['tokens_sem_sw'][:7]}")

Documentos processados: 32

Exemplo - Documento 1:
  Original : O presidente Luiz Inácio Lula da Silva assinou hoje em Brasília um pac...
  Tokens   : ['o', 'presidente', 'luiz', 'inácio', 'lula', 'da', 'silva']
  Sem SW   : ['presidente', 'luiz', 'inácio', 'lula', 'silva', 'assinou', 'hoje']


## 5 Estatísticas e Salvamento

In [ ]:
total_tokens = sum(len(d['tokens']) for d in docs_processados)
total_sem_sw = sum(len(d['tokens_sem_sw']) for d in docs_processados)

print(f"Total documentos : {len(docs_processados)}")
print(f"Total tokens     : {total_tokens}")
print(f"Total sem SW     : {total_sem_sw}")
print(f"Média tokens/doc : {total_tokens/len(docs_processados):.1f}")

import pandas as pd
df_out = pd.DataFrame([{
    'id': d['id'], 'original': d['original'],
    'limpo': d['limpo'], 'tokens_sem_sw': ' '.join(d['tokens_sem_sw'])
} for d in docs_processados])
df_out.to_csv(DIRETORIO_LOCAL + 'dataset.csv', sep=';', index=False, encoding='utf-8')
print(f"\ndataset.csv salvo em {DIRETORIO_LOCAL}")

Total documentos : 32
Total tokens     : 665
Total sem SW     : 394
Média tokens/doc : 20.8

dataset.csv salvo em /content/SRI/


In [ ]:
import time, datetime
final = time.time()
tempo = str(datetime.timedelta(seconds=int(round(final - inicio_processamento))))
print(f"Tempo de processamento: {tempo} (h:mm:ss)")

Tempo de processamento: 0:00:03 (h:mm:ss)
